In [1]:
import pandas as pd

customer_clustering = pd.read_csv('data/customer_clustering.csv')
customer_clustering.head(), customer_clustering.shape

(       월평균값  월중앙값  월최댓값  월최솟값  회원기간  cluster
 0  4.833333   5.0     8     2    47        2
 1  5.083333   5.0     7     3    47        2
 2  4.583333   5.0     6     3    47        2
 3  4.833333   4.5     7     2    47        2
 4  3.916667   4.0     6     1    47        2,
 (4192, 6))

In [2]:
customer_join = pd.read_csv('data/customer_join.csv')
customer_join.shape
customer_join.head()


,customer_id,name,class,gender,start_date,end_date,campaign_id,is_deleted,class_name,price,...,max_x,min_x,routine_flg_x,mean_y,median_y,max_y,min_y,routine_flg_y,calc_date,membership_period
0,OA832399,XXXX,C01,F,2015-05-01,NaN,CA1,0,0_종일,10500,...,8,2,1,4.833333,5.0,8,2,1,2019-04-30,47
1,PL270116,XXXXX,C01,M,2015-05-01,NaN,CA1,0,0_종일,10500,...,7,3,1,5.083333,5.0,7,3,1,2019-04-30,47
2,OA974876,XXXXX,C01,M,2015-05-01,NaN,CA1,0,0_종일,10500,...,6,3,1,4.583333,5.0,6,3,1,2019-04-30,47
3,HD024127,XXXXX,C01,F,2015-05-01,NaN,CA1,0,0_종일,10500,...,7,2,1,4.833333,4.5,7,2,1,2019-04-30,47
4,HD661448,XXXXX,C03,F,2015-05-01,NaN,CA1,0,2_야간,6000,...,6,1,1,3.916667,4.0,6,1,1,2019-04-30,47


In [3]:
customer_clustering = pd.concat([customer_clustering, customer_join], axis=1)
customer_clustering.head(2)

,월평균값,월중앙값,월최댓값,월최솟값,회원기간,cluster,customer_id,name,class,gender,...,max_x,min_x,routine_flg_x,mean_y,median_y,max_y,min_y,routine_flg_y,calc_date,membership_period
0,4.833333,5.0,8,2,47,2,OA832399,XXXX,C01,F,...,8,2,1,4.833333,5.0,8,2,1,2019-04-30,47
1,5.083333,5.0,7,3,47,2,PL270116,XXXXX,C01,M,...,7,3,1,5.083333,5.0,7,3,1,2019-04-30,47


In [4]:
# 탈퇴 기준 
customer_clustering.groupby(['cluster','is_deleted'], as_index=False).count()[['cluster','is_deleted','customer_id']]
# customer_clustering.groupby(
#     ['cluster', 'is_deleted'],
#     as_index=False
# )['customer_id'].count().rename(columns={'customer_id': 'count'})


,cluster,is_deleted,customer_id
0,0,0,791
1,0,1,543
2,1,1,771
3,2,0,1231
4,2,1,18
5,3,0,820
6,3,1,18


-> 0,1번 그룹 관리 필요

In [5]:
# routine 기준 
customer_clustering.groupby(['cluster','routine_flg_x'], as_index=False).count()[['cluster','routine_flg_x','customer_id']]

,cluster,routine_flg_x,customer_id
0,0,0,227
1,0,1,1107
2,1,0,499
3,1,1,272
4,2,0,2
5,2,1,1247
6,3,0,51
7,3,1,787


In [6]:
summary = customer_clustering.groupby('cluster').agg({
    'customer_id':'count',
    'is_deleted':'mean',
    'routine_flg_x':'mean',
    '회원기간':'mean',
    '월평균값':'mean'
}).round(2)
summary

,customer_id,is_deleted,routine_flg_x,회원기간,월평균값
cluster,,,,,
0,1334,0.41,0.83,14.86,5.54
1,771,1.00,0.35,9.28,3.07
2,1249,0.01,1.00,36.92,4.68
3,838,0.02,0.94,7.02,8.06


In [7]:
summary.columns = ['인원','탈퇴율','정기비율','회원기간(개월)','월평균이용']
summary

,인원,탈퇴율,정기비율,회원기간(개월),월평균이용
cluster,,,,,
0,1334,0.41,0.83,14.86,5.54
1,771,1.00,0.35,9.28,3.07
2,1249,0.01,1.00,36.92,4.68
3,838,0.02,0.94,7.02,8.06


In [8]:
summary['페르소나'] = [
    '혼합 그룹(탈퇴 41%)',
    '전원 탈퇴',
    '장기 안정 단골(탈퇴 1%)',
    '신규의욕+지속(탈퇴 2%)'
]

summary

,인원,탈퇴율,정기비율,회원기간(개월),월평균이용,페르소나
cluster,,,,,,
0,1334,0.41,0.83,14.86,5.54,혼합 그룹(탈퇴 41%)
1,771,1.00,0.35,9.28,3.07,전원 탈퇴
2,1249,0.01,1.00,36.92,4.68,장기 안정 단골(탈퇴 1%)
3,838,0.02,0.94,7.02,8.06,신규의욕+지속(탈퇴 2%)


In [9]:
customer_clustering.to_csv('./data/customer_clustering2.csv',index=False)


### '이번달까지의 이용 패턴으로 다음 달 이용 횟수 예측 가능?'
#### 지도학습 -> 회귀(예측), 분류 

- 과거의 데이터 (시간의 흐름에 따른 이용 횟수 패턴)
- 2018년 11월: 종속변수(y)
- 2018년 5월 ~ 2018년 10월 : 독립변수(X)


In [10]:
uselog = pd.read_csv('data/use_log.csv')
uselog.shape

(197428, 3)

In [11]:
uselog.info()

<class 'pandas.DataFrame'>
RangeIndex: 197428 entries, 0 to 197427
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   log_id       197428 non-null  str  
 1   customer_id  197428 non-null  str  
 2   usedate      197428 non-null  str  
dtypes: str(3)
memory usage: 4.5 MB


In [12]:
uselog['usedate'] = pd.to_datetime(uselog['usedate'])
uselog.info()

<class 'pandas.DataFrame'>
RangeIndex: 197428 entries, 0 to 197427
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   log_id       197428 non-null  str           
 1   customer_id  197428 non-null  str           
 2   usedate      197428 non-null  datetime64[us]
dtypes: datetime64[us](1), str(2)
memory usage: 4.5 MB


In [13]:
uselog['연월'] = uselog['usedate'].dt.strftime('%Y%m')
uselog.head()

,log_id,customer_id,usedate,연월
0,L00000049012330,AS009373,2018-04-01,201804
1,L00000049012331,AS015315,2018-04-01,201804
2,L00000049012332,AS040841,2018-04-01,201804
3,L00000049012333,AS046594,2018-04-01,201804
4,L00000049012334,AS073285,2018-04-01,201804


In [14]:
uselog_months = uselog.groupby(['연월','customer_id'],as_index=False).count()
uselog_months.head()

,연월,customer_id,log_id,usedate
0,201804,AS002855,4,4
1,201804,AS009013,2,2
2,201804,AS009373,3,3
3,201804,AS015315,6,6
4,201804,AS015739,7,7


In [15]:
uselog_months.rename(columns={'log_id':'count'},inplace=True)
uselog_months.head()

,연월,customer_id,count,usedate
0,201804,AS002855,4,4
1,201804,AS009013,2,2
2,201804,AS009373,3,3
3,201804,AS015315,6,6
4,201804,AS015739,7,7


In [16]:
del uselog_months['usedate']
uselog_months.head()

,연월,customer_id,count
0,201804,AS002855,4
1,201804,AS009013,2
2,201804,AS009373,3
3,201804,AS015315,6
4,201804,AS015739,7


In [18]:
year_months = list(uselog_months['연월'].unique())
year_months

['201804',
 '201805',
 '201806',
 '201807',
 '201808',
 '201809',
 '201810',
 '201811',
 '201812',
 '201901',
 '201902',
 '201903']

In [19]:
predict_data = pd.DataFrame()

In [28]:
for i in range(6,len(year_months)):
    tmp = uselog_months.loc[uselog_months['연월']==year_months[i]]
    tmp.rename(columns={'count':'count_pred'},inplace=True)
    print(i, year_months[i], tmp.columns)
    for j in range(1, 7):
        tmp_before = uselog_months.loc[uselog_months['연월']==year_months[i-j]]
        tmp.rename(columns={'count':'count_{}'.format(i-j)},inplace=True)
        del tmp_before['연월']
        tmp = pd.merge(tmp,tmp_before,how='left',on = 'customer_id')
        # print(year_months[i-j],tmp_before.columns)
    predict_data = pd.concat([predict_data, tmp],ignore_index=True)

6 201810 Index(['연월', 'customer_id', 'count_pred'], dtype='str')
7 201811 Index(['연월', 'customer_id', 'count_pred'], dtype='str')
8 201812 Index(['연월', 'customer_id', 'count_pred'], dtype='str')
9 201901 Index(['연월', 'customer_id', 'count_pred'], dtype='str')
10 201902 Index(['연월', 'customer_id', 'count_pred'], dtype='str')
11 201903 Index(['연월', 'customer_id', 'count_pred'], dtype='str')


In [29]:
predict_data.shape

(39573, 14)

In [25]:
predict_data.head()

,연월,customer_id,count_pred,count_4,count_3,count_2,count_1,count_0,count,count_5,count_6,count_7,count_8,count_9
0,201810,AS002855,3,7.0,3.0,5.0,5.0,5.0,4.0,NaN,NaN,NaN,NaN,NaN
1,201810,AS008805,2,2.0,5.0,7.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,201810,AS009373,5,6.0,6.0,7.0,4.0,4.0,3.0,NaN,NaN,NaN,NaN,NaN
3,201810,AS015233,7,9.0,11.0,5.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN
4,201810,AS015315,4,7.0,3.0,6.0,3.0,3.0,6.0,NaN,NaN,NaN,NaN,NaN
